In [1]:
import pandas as pd
import numpy as np
import os
import json
import glob

In [2]:
# Define paths
DATA_DIR = '../data'
IP_INFO_PATH = os.path.join(DATA_DIR, 'ip_info.csv')
ISP_COMPLETE_PATH = os.path.join(DATA_DIR, 'isp_complete.csv')

print(f"Reading {IP_INFO_PATH}...")
ip_info_df = pd.read_csv(IP_INFO_PATH)
print(f"Loaded {len(ip_info_df)} rows.")

Reading ../data\ip_info.csv...


C:\Users\user\AppData\Local\Temp\ipykernel_28936\3085779706.py:7: DtypeWarning: Columns (0: mobile) have mixed types. Specify dtype option on import or set low_memory=False.
  ip_info_df = pd.read_csv(IP_INFO_PATH)


Loaded 9449557 rows.


In [3]:
# Clean up mobile column mapping
ip_info_df["mobile"] = ip_info_df["mobile"].map(
    {"True": True, "False": False, True: True, False: False}
)

# Calculate mobile test ratio per ISP
isp_stats = ip_info_df.groupby("isp").agg(
    total_tests=("client_ip", "count"),
    mobile_tests=("mobile", lambda x: sum(x == True))
).reset_index()

isp_stats["mobile_ratio"] = isp_stats["mobile_tests"] / isp_stats["total_tests"]

In [4]:
def classify_isp(row):
    isp_name = str(row["isp"]).strip()
    isp_lower = isp_name.lower()
    
    # Baseline classification based on mobile ratio
    if row["mobile_ratio"] >= 0.8:
        category = "Mobile"
        network_type = "cellular"
        description = "Mobile network"
    else:
        network_type = "broadband"
        category = "Others"
        description = "Other or unclassifiable"
        
    # Refine classification based on keyword matching in ISP name
    if any(k in isp_lower for k in ["university", "school", "college", "institute", "academy", "wittaya"]):
        category = "Educational"
        description = "Educational institution"
    elif any(k in isp_lower for k in ["government", "embassy", "ministry", "hospital", "authority", "department"]):
        category = "Government"
        description = "Government agency"
    elif any(k in isp_lower for k in ["hotel", "resort", "inn", "plaza", "suites"]):
        category = "Hotel & Resort"
        description = "Hotel or resort"
    elif any(k in isp_lower for k in ["telecom", "network", "idc", "datacenter", "communication", "solution"]):
        category = "Telco & Solution Providers"
        description = "Telecommunications and technology solution provider"
    elif any(k in isp_lower for k in ["gateway", "m247", "amazon", "alibaba", "cloudflare", "google", "microsoft"]):
        category = "International Providers & Gateway"
        description = "International service provider or data gateway"
    elif any(k in isp_lower for k in ["ais-fibre", "3bb", "true online", "tot public", "triple t", "broadband"]):
        category = "Consumer Broadband"
        description = "Consumer broadband"
    elif any(k in isp_lower for k in ["co., ltd", "company limited", "plc", "corp", "bank"]):
        category = "Broadband Enterprise"
        description = "Private enterprise network"
        
    return pd.Series([category, description, network_type])

print("Classifying ISPs based on rules...")
isp_stats[["category", "description", "network_type"]] = isp_stats.apply(classify_isp, axis=1)

Classifying ISPs based on rules...


In [5]:
isp_complete_df = isp_stats[["isp", "category", "description", "network_type"]]
isp_complete_df = isp_complete_df.sort_values("isp").reset_index(drop=True)
isp_complete_df.to_csv(ISP_COMPLETE_PATH, index=False)
print(f"Processed {len(isp_complete_df)} unique ISPs. Saved to {ISP_COMPLETE_PATH}!")

Processed 1360 unique ISPs. Saved to ../data\isp_complete.csv!


In [6]:
print("Creating client mapping DataFrame in Pandas...")
client_df = ip_info_df.merge(isp_complete_df, on='isp', how='inner')[['client_ip', 'isp', 'category', 'network_type']]
client_df_indexed = client_df.set_index('client_ip')
print(f"Created indexed DataFrame for {len(client_df_indexed):,} unique client IPs.")

# Clear variables to free memory
del ip_info_df, isp_complete_df, client_df

Creating client mapping DataFrame in Pandas...


Created indexed DataFrame for 9,449,541 unique client IPs.


In [7]:
# Define input directories
DATA_DIR = '../data'
MLAB_RESULTS_DIR = os.path.join(DATA_DIR, 'mlab_results')
MLAB_RESULTS_2023_DIR = os.path.join(DATA_DIR, 'mlab_results_2023_gap')
MLAB_RESULTS_2025_DIR = os.path.join(DATA_DIR, 'mlab_results_2025_gap')

# Create final output directory
output_dir = os.path.join(DATA_DIR, 'processed_results.parquet')
os.makedirs(output_dir, exist_ok=True)

# Get all parquet files to process
files_main = glob.glob(os.path.join(MLAB_RESULTS_DIR, '*.parquet'))
files_2023 = glob.glob(os.path.join(MLAB_RESULTS_2023_DIR, '*.parquet'))
files_2025 = glob.glob(os.path.join(MLAB_RESULTS_2025_DIR, '*.parquet'))
all_files = files_main + files_2023 + files_2025

print(f"Total parquet files to process: {len(all_files)}")

Total parquet files to process: 313


In [8]:
# Load city_to_province mapping
CITY_TO_PROVINCE_PATH = os.path.join(DATA_DIR, 'city_to_province.json')
with open(CITY_TO_PROVINCE_PATH, encoding='utf-8') as json_data:
    non_province_mapping = json.load(json_data)
non_province_mapping = {k.strip().title(): v for k, v in non_province_mapping.items()}

In [9]:
def process_file(file_path, output_path, mapping_dict, client_df_indexed):
    # Read file using Pandas
    df = pd.read_parquet(file_path)
    
    # 1. Keep original city name (cleaned to Title Case)
    df["city"] = df["city"].str.strip().str.title()
    df = df.dropna(subset=["city"])
    
    # 2. Map city to province and store as new column 'province'
    df["province"] = df["city"].map(mapping_dict).fillna(df["city"])

    # 3. Filter throughput > 0
    df = df[df["mean_throughput_mbps"] > 0]
    if len(df) == 0:
        return
        
    # 4. Join client IP metadata (extremely fast left join on indexed DataFrame)
    df = df.join(client_df_indexed, on='client_ip', how='left')
    
    # 5. Extract temporal features: date column contains date objects, test_time contains datetime64
    date_dt = pd.to_datetime(df['date'])
    df['hour'] = df['test_time'].dt.hour
    df['year'] = date_dt.dt.year
    df['month'] = date_dt.dt.month
    
    # 6. Keep target columns (including client_ip, city, and new province column)
    final_columns = [
        'date', 'test_time', 'mean_throughput_mbps', 'min_rtt', 'loss_rate',
        'client_ip', 'city', 'province', 'latitude', 'longitude', 'type',
        'isp', 'network_type', 'category', 'hour', 'year', 'month'
    ]
    df_final = df[final_columns]
    
    # Save as parquet
    df_final.to_parquet(output_path, index=False)

In [10]:
print("Processing all files sequentially using Pandas...")
for idx, f in enumerate(all_files):
    out_name = f"part_{idx}.parquet"
    out_path = os.path.join(output_dir, out_name)
    process_file(f, out_path, non_province_mapping, client_df_indexed)
    if (idx + 1) % 20 == 0 or (idx + 1) == len(all_files):
        print(f"Processed {idx + 1}/{len(all_files)} files...")
        
print("Data preparation successfully completed!")

Processing all files sequentially using Pandas...


Processed 20/313 files...


Processed 40/313 files...


Processed 60/313 files...


Processed 80/313 files...


...

Processed 240/313 files...


Processed 260/313 files...


Processed 280/313 files...


Processed 300/313 files...


Processed 313/313 files...
Data preparation successfully completed!
